# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5f21a6d0-0828-4f34-b2d9-0268f68bdac3;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central


	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central


	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central


	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central


	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central


... (4 outputs left out)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/17 16:05:02 ERROR Inbox: Ignoring error
java.lang.NullPointerException: Cannot invoke "org.apache.spark.storage.BlockManagerId.executorId()" because "idWithoutTopologyInfo" is null
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:696)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:141)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:104)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:101)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop$$receiveLoop(MessageLoop.scala:76)
	at org.apache.spark.rpc.netty.MessageLoop$$anon$1.run(MessageLoop.scala:42)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144)
	at java.base/java.util.concurrent.T

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [2]:
# Déclaration des tables Bronze du dataset LinkedIn Job Postings à croiser
BRONZE_TABLES: dict[str, str] = {
    "main":       "/lakehouse/bronze/postings",    # Table de faits principale (offres d'emploi)
    "companies":  "/lakehouse/bronze/companies",   # Dimension entreprises (company_id)
    "salaries":   "/lakehouse/bronze/salaries",    # Table secondaire des salaires détaillés (job_id)
    "job_skills": "/lakehouse/bronze/job_skills",  # Table secondaire des compétences requises (job_id)
}

assert len(BRONZE_TABLES) > 0, "Renseignez vos chemins Delta Bronze dans `BRONZE_TABLES`."
bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}
for alias, df in bronze_dfs.items():
    print(f"[{alias:<10}] {BRONZE_TABLES[alias]:<30} -> {df.count():>10,} lignes | colonnes : {df.columns[:7]}")

# Diagnostic de qualité (valeurs nulles et doublons) sur la table principale `postings`
df_bronze = bronze_dfs.get("main")
key_cols = [
    "job_id", "company_id", "company_name", "title", "location",
    "formatted_work_type", "formatted_experience_level", "normalized_salary", "views", "applies"
]
null_report = df_bronze.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)).alias(c)
    for c in key_cols
])
assert null_report is not None, "Calculez le diagnostic de valeurs nulles `null_report`."
print("\n=== Diagnostic des valeurs NULL / vides sur la table principale (postings) ===")
null_report.show(truncate=False)

total_rows = df_bronze.count()
distinct_jobs = df_bronze.select("job_id").distinct().count()
print(f"Total lignes Bronze : {total_rows:,} | job_id distincts : {distinct_jobs:,} | Doublons : {total_rows - distinct_jobs:,}")


26/11/17 16:05:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[main      ] /lakehouse/bronze/postings     ->    123,849 lignes | colonnes : ['job_id', 'company_name', 'title', 'max_salary', 'pay_period', 'location', 'company_id']


[companies ] /lakehouse/bronze/companies    ->     24,473 lignes | colonnes : ['company_id', 'name', 'company_size', 'state', 'country', 'city', 'zip_code']


[salaries  ] /lakehouse/bronze/salaries     ->     40,785 lignes | colonnes : ['salary_id', 'job_id', 'max_salary', 'med_salary', 'min_salary', 'pay_period', 'currency']


[job_skills] /lakehouse/bronze/job_skills   ->    213,768 lignes | colonnes : ['job_id', 'skill_abr', '_ingested_at', '_source_dataset', '_source_file']



=== Diagnostic des valeurs NULL / vides sur la table principale (postings) ===


+------+----------+------------+-----+--------+-------------------+--------------------------+-----------------+-----+-------+
|job_id|company_id|company_name|title|location|formatted_work_type|formatted_experience_level|normalized_salary|views|applies|
+------+----------+------------+-----+--------+-------------------+--------------------------+-----------------+-----+-------+
|0     |1717      |1719        |0    |0       |0                  |29409                     |87776            |1689 |100529 |
+------+----------+------------+-----+--------+-------------------+--------------------------+-----------------+-----+-------+



Total lignes Bronze : 123,849 | job_id distincts : 123,849 | Doublons : 0


## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [3]:
SILVER_MAIN_PATH = "/lakehouse/silver/linkedin_jobs_enriched"

# 1) Nettoyage, filtrage d'anomalies et typage fort de la table principale `postings`
df_postings_clean = (
    bronze_dfs["main"]
    .filter(F.col("job_id").isNotNull() & F.col("job_id").rlike("^[0-9]+$"))
    .dropDuplicates(["job_id"])
    .withColumn("company_id", F.regexp_replace(F.trim(F.col("company_id")), r"\.0$", ""))
    .withColumn("listed_at", F.to_timestamp(F.col("original_listed_time").cast("double") / 1000.0))
    .withColumn("listed_date", F.to_date(F.col("listed_at")))
    .withColumn("views_num", F.coalesce(F.col("views").cast("double").cast("long"), F.lit(0)))
    .withColumn("applies_num", F.coalesce(F.col("applies").cast("double").cast("long"), F.lit(0)))
    .withColumn("post_norm_salary", F.col("normalized_salary").cast("double"))
    .withColumn("work_type_clean", F.coalesce(F.upper(F.trim(F.col("formatted_work_type"))), F.lit("OTHER")))
    .withColumn("experience_level", F.coalesce(F.upper(F.trim(F.col("formatted_experience_level"))), F.lit("NOT_SPECIFIED")))
    .withColumn("is_remote", F.when(F.col("remote_allowed").isin("1", "1.0", "true", "True"), F.lit(True)).otherwise(F.lit(False)))
)
count_before_join = df_postings_clean.count()

# 2) Pré-agrégation / dédoublonnage des sources secondaires pour garantir l'unicité des clés (0 explosion N-M)
df_companies_dim = (
    bronze_dfs["companies"]
    .withColumn("company_id", F.regexp_replace(F.trim(F.col("company_id")), r"\.0$", ""))
    .filter(F.col("company_id").isNotNull() & F.col("company_id").rlike("^[0-9]+$"))
    .dropDuplicates(["company_id"])
    .select(
        "company_id",
        F.trim(F.col("name")).alias("company_official_name"),
        F.col("company_size").cast("double").cast("int").alias("company_size_tier"),
        F.upper(F.trim(F.col("country"))).alias("company_country"),
        F.trim(F.col("state")).alias("company_state"),
        F.trim(F.col("city")).alias("company_city"),
    )
)

df_salaries_agg = (
    bronze_dfs["salaries"]
    .filter(F.col("job_id").isNotNull() & F.col("job_id").rlike("^[0-9]+$"))
    .withColumn("min_s", F.col("min_salary").cast("double"))
    .withColumn("med_s", F.col("med_salary").cast("double"))
    .withColumn("max_s", F.col("max_salary").cast("double"))
    .withColumn("base_s", F.coalesce(F.col("med_s"), (F.col("min_s") + F.col("max_s")) / 2.0, F.col("max_s"), F.col("min_s")))
    .withColumn(
        "sal_annualized",
        F.when(F.upper(F.col("pay_period")) == "HOURLY", F.col("base_s") * 2080.0)
         .when(F.upper(F.col("pay_period")) == "MONTHLY", F.col("base_s") * 12.0)
         .when(F.upper(F.col("pay_period")) == "WEEKLY", F.col("base_s") * 52.0)
         .when(F.upper(F.col("pay_period")) == "BIWEEKLY", F.col("base_s") * 26.0)
         .otherwise(F.col("base_s"))
    )
    .groupBy("job_id")
    .agg(
        F.round(F.avg("sal_annualized"), 2).alias("sal_table_annual_usd"),
        F.first("pay_period", ignorenulls=True).alias("sal_pay_period"),
        F.first("currency", ignorenulls=True).alias("sal_currency"),
    )
)

df_skills_agg = (
    bronze_dfs["job_skills"]
    .filter(F.col("job_id").isNotNull() & F.col("job_id").rlike("^[0-9]+$") & F.col("skill_abr").isNotNull())
    .groupBy("job_id")
    .agg(
        F.countDistinct("skill_abr").alias("skills_count"),
        F.concat_ws(",", F.sort_array(F.collect_set("skill_abr"))).alias("skills_list"),
        F.first("skill_abr").alias("primary_skill_abr"),
    )
)

# 3) Jointure distribuée (BroadcastHashJoin sur les tables de dimension/pré-agrégées) & Colonnes dérivées métier
raw_annual_salary = F.coalesce(F.col("post_norm_salary"), F.col("sal_table_annual_usd"))

df_silver = (
    df_postings_clean
    .join(F.broadcast(df_companies_dim), on="company_id", how="left")
    .join(F.broadcast(df_salaries_agg), on="job_id", how="left")
    .join(F.broadcast(df_skills_agg), on="job_id", how="left")
    .withColumn("company_name_clean", F.coalesce(F.col("company_official_name"), F.trim(F.col("company_name")), F.lit("Unknown Company")))
    .withColumn("skills_count", F.coalesce(F.col("skills_count"), F.lit(0)))
    .withColumn("primary_skill_abr", F.coalesce(F.col("primary_skill_abr"), F.lit("UNSPECIFIED")))
    .withColumn(
        "annual_salary_usd",
        F.when((raw_annual_salary >= 10000.0) & (raw_annual_salary <= 1000000.0), F.round(raw_annual_salary, 2)).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "application_conversion_rate",
        F.when(F.col("views_num") > 0, F.round((F.col("applies_num") / F.col("views_num")) * 100.0, 2)).otherwise(F.lit(0.0))
    )
    .withColumn(
        "salary_per_skill_usd",
        F.when(F.col("annual_salary_usd").isNotNull() & (F.col("skills_count") > 0), F.round(F.col("annual_salary_usd") / F.col("skills_count"), 2))
         .otherwise(F.col("annual_salary_usd"))
    )
    .select(
        "job_id", "company_id", "company_name_clean", "title", "location",
        "company_country", "company_state", "company_city", "company_size_tier",
        "work_type_clean", "experience_level", "is_remote",
        "listed_at", "listed_date",
        "views_num", "applies_num", "application_conversion_rate",
        "annual_salary_usd", "salary_per_skill_usd",
        "skills_count", "primary_skill_abr", "skills_list"
    )
)

assert df_silver is not None, "Construisez le DataFrame nettoyé et joint `df_silver`."
print("=== Plan d'exécution physique Spark (BroadcastHashJoin) ===")
df_silver.explain("simple")

# 4) Sauvegarde au format Delta dans `/lakehouse/silver/linkedin_jobs_enriched`
OVERWRITE_EXISTING = False  # Mettre à True pour forcer la réécriture complète de la table Silver
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = spark._jvm.org.apache.hadoop.fs.Path
if OVERWRITE_EXISTING or not fs.exists(PathHDFS(f"{SILVER_MAIN_PATH}/_delta_log")):
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_MAIN_PATH)

df_silver_loaded = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH)
silver_count = df_silver_loaded.count()
if count_before_join is not None:
    assert silver_count <= count_before_join, (
        f"Alerte explosion N-M : {silver_count:,} lignes après jointure > {count_before_join:,} lignes avant jointure !"
    )

matched_companies = df_silver_loaded.filter(F.col("company_country").isNotNull()).count()
matched_skills = df_silver_loaded.filter(F.col("skills_count") > 0).count()
matched_salaries = df_silver_loaded.filter(F.col("annual_salary_usd").isNotNull()).count()
print(f"\n[SILVER OK] Table écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes (avant jointure : {count_before_join:,})")
print(f" - Taux de rapprochement Entreprises (companies) : {matched_companies / silver_count * 100:.1f}% ({matched_companies:,} lignes)")
print(f" - Taux de rapprochement Compétences (job_skills) : {matched_skills / silver_count * 100:.1f}% ({matched_skills:,} lignes)")
print(f" - Offres avec salaire annuel normalisé valide   : {matched_salaries / silver_count * 100:.1f}% ({matched_salaries:,} lignes)")


=== Plan d'exécution physique Spark (BroadcastHashJoin) ===


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [job_id#0, company_id#2086, company_name_clean#2322, title#2331, location#2337, company_country#2279, company_state#2280, company_city#2281, company_size_tier#2278, work_type_clean#2092, experience_level#2093, is_remote#2094, listed_at#2087, listed_date#2088, views_num#2089L, applies_num#2090L, application_conversion_rate#2326, annual_salary_usd#2325, CASE WHEN (isnotnull(annual_salary_usd#2325) AND (skills_count#2323L > 0)) THEN round((annual_salary_usd#2325 / cast(skills_count#2323L as double)), 2) ELSE annual_salary_usd#2325 END AS salary_per_skill_usd#2327, skills_count#2323L, primary_skill_abr#2324, skills_list#2312]
   +- Project [job_id#0, company_id#2086, title#2331, location#2337, listed_at#2087, listed_date#2088, views_num#2089L, applies_num#2090L, work_type_clean#2092, experience_level#2093, is_remote#2094, company_size_tier#2278, company_country#2279, company_state#2280, company_city#2281, coalesce(skills_co


[SILVER OK] Table écrite dans /lakehouse/silver/linkedin_jobs_enriched : 123,849 lignes (avant jointure : 123,849)
 - Taux de rapprochement Entreprises (companies) : 98.6% (122,131 lignes)
 - Taux de rapprochement Compétences (job_skills) : 98.6% (122,096 lignes)
 - Offres avec salaire annuel normalisé valide   : 28.8% (35,618 lignes)


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [4]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794930311489,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"9fd81c2a-ff19-44ca-95e6-633d8bfa3931","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-7b2b6938-57ca-4346-b24a-9955d053b750-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-8908d910-a984-4d5b-aac3-1c6c91666adc-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-d14cc583-752e-4f10-84cd-53408678580c-c000.snappy.parquet","partition

Schema Enforcement validé — écriture invalide bloquée : [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.


### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [5]:
# Upsert (MERGE INTO) sur `SILVER_MAIN_PATH` puis comparaison Time Travel (versionAsOf=0 vs version courante)
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")

if history_df.count() < 2:
    # Construction d'un lot correctif : 2 lignes existantes mises à jour (+100 vues) + 1 nouvelle offre synthétique
    existing_sample = (
        spark.read.format("delta").load(SILVER_MAIN_PATH)
        .limit(2)
        .withColumn("views_num", F.col("views_num") + F.lit(100))
    )
    new_row = (
        spark.read.format("delta").load(SILVER_MAIN_PATH)
        .limit(1)
        .withColumn("job_id", F.lit("9999999999"))
        .withColumn("title", F.lit("Senior Big Data & Lakehouse Engineer (Test MERGE INTO)"))
        .withColumn("annual_salary_usd", F.lit(165000.0))
    )
    raw_updates = existing_sample.unionByName(new_row)
    updates_df = spark.createDataFrame(raw_updates.collect(), schema=raw_updates.schema)

    (
        target_delta.alias("target")
        .merge(updates_df.alias("source"), "target.job_id = source.job_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")

history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"

count_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH).count()
count_latest = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
print(f"Time Travel Delta — Version 0 : {count_v0:,} lignes | Version courante (après MERGE) : {count_latest:,} lignes (+{count_latest - count_v0} offre insérée)")


+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      1|2026-11-17 15:46:34.489|    MERGE|{numTargetRowsCopied -> 30758, numTargetRowsDeleted -> 0, numTargetBytesRemov...|
|      0|2026-11-17 15:45:13.702|    WRITE|{numFiles -> 4, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> ...|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+



Time Travel Delta — Version 0 : 123,849 lignes | Version courante (après MERGE) : 123,850 lignes (+1 offre insérée)
